In [2]:
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv('env_path')

api_key = os.getenv("api_key")
csv_file = "loan_applications.csv"
if os.path.exists(csv_file):
    os.remove(csv_file)

if not os.path.exists(csv_file):

    df = pd.DataFrame(columns=[
        "id",
        "full_name",
        "email",
        "post",
        "company_name",
        "salary_per_year",
        "loan_amount",
        "loan_eligibility",
        "allowed"
    ])

    df.to_csv(csv_file, index=False)

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel
from datetime import datetime
class Employee(BaseModel):
    full_name: str
    email: str
    post: str
    company_name: str
    salary_per_year: float
    loan_amount: float
    loan_eligibility: str
    error_message: str

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=api_key
)

structured_llm = llm.with_structured_output(Employee)

system_message = '''You are a helpful and a wise bank loan approver.
You will revcieve prompts and messages that will contain salary and employment details and loan amount asked.

Salary detail should be salary 'per year' or 'py', if the salary is provided per month or 'pm',
then convert the salary into per year by multiplying the per month salary by 12.
Salary should always be in Indian Rupee. Any amount is assumed to be Indian Rupee.

Please give the salary output as decimal value with round off to 2 decimal digits.

Employment details shall include full name, email, post that they work on and company name.

A full name is text that contains a title and at least 2 words. If full name not provided, its an error
Return full name as 1 single string like 'Title Full Name'
Please validate the email, if its in correct format or not. If email not provided or wrong, its an error

No rules for post and company name. If post and company name not provided, its an error.

Everything that is asked is REQUIRED and MANDATORY.
Failure to provide any data, should return a message that says "Error-" followed by the error.

Loan Eligibility Criteria:
Loan is ONLY and ONLY approved (True State) if and only if loan amount asked is less than or equal to 50% or half of yearly salary.
Return Eligibility Critera as T for True or F for False.

If no error keep this error message empty.
Return everything (that includes salary info and employee details, eligibity and error message) in 1 json entry.

for example
{{
  "full_name": "Mr. Ankur Arora",
  "email": "aroraankur004@gmail.com",
  "post": "Software Engineer",
  "company_name": "Wissen Technologies",
  "salary_per_year": 600000.00,
  "loan_amount": 20000.00,
  "loan_eligibility": "T",
  "error_message": ""
}}

No other text or words or any string other than a response like this.
'''
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        ("human", "Help and check if this person is eligible for loan. {Human}")
    ]
)

chain = prompt | structured_llm

human = input()
response = chain.invoke({
    "Human": human
})

new_record = response.model_dump()

if new_record["error_message"]:

    print(new_record["error_message"])

else:

    del new_record["error_message"]


    unique_id = (
        f"{new_record['full_name'].split()[1]}_"
        f"{new_record['post'].replace(' ','')}_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    )

    new_record["allowed"] = "Pending"
    new_record["id"] = unique_id

    COLUMN_ORDER = [
        "id",
        "full_name",
        "email",
        "post",
        "company_name",
        "salary_per_year",
        "loan_amount",
        "loan_eligibility",
        "allowed"
    ]

    pd.DataFrame([new_record])[COLUMN_ORDER].to_csv(
        "loan_applications.csv",
        mode="a",
        header=False,
        index=False
    )
    print("Successfully entered your request in our database")

Mr. Ankur Arora ankuraroratest@gmail.com software engineer at Google earning 12 lakhs per annum demanding a loan of 40k.
Successfully entered your request in our database
